# Classical Model Training Using the CONNIE Dataset (multiclass with image descriptors)

In [12]:
%run ./../notebook_init.py

import os
import uproot
import optuna
import mlflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from glob import glob
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix,
                             precision_score,
                             recall_score,
                             f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from scipy.ndimage import convolve

from core import DATA_FOLDER, RESULTS_FOLDER, OUTPUT_FOLDER
from scripts.connie_training_utils import (Seed, extract_skeleton_features,
                                           extract_fourier_descriptors,
                                           create_mask_from_coords,
                                           find_contours, skeletonize)

In [13]:
train_data = os.path.join(DATA_FOLDER, "train_data_root_full")

In [14]:
seed = Seed()

In [15]:
#categories = ["Alpha", "Blob", "Diffusion_Hit", "Electron", "Muon", "Others"]
categories = ["Blob", "Diffusion Hit", "Electron", "Muon", "Others"]

branch_name = "hitSumm"

In [16]:
all_data_list = []
all_data_list_excluded_vars = []

print("Starting data loading")
for category in categories:
    category_path = os.path.join(train_data, category)
    root_files = glob(os.path.join(category_path, "*.root"))

    if not root_files:
        print(f"Warning: No .root files found in {category_path}")
        continue

    print(f"Processing category: {category} ({len(root_files)} files)")
    for idx, file_path in enumerate(root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                file_branch = file[branch_name]
                df = file_branch.arrays(library="pd")
                df['label'] = category
                all_data_list.append(df)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")


Starting data loading
Processing category: Blob (351 files)
Processing category: Diffusion Hit (44 files)
Processing category: Electron (366 files)
Processing category: Muon (2596 files)
Processing category: Others (220 files)


Combine all DataFrames into a single DataFrame

In [17]:
if all_data_list:
    df_combined = pd.concat(all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(df_combined)} rows of data.")
else:
    print("No data loaded.")

Successfully loaded 3577 rows of data.


* Calculate mean of ePix and level to be used as features
* Remove features with more than one dimension, such as xPix and yPix
* Remove "flag", as we already filtered for only valid events
* Drop columns with no variance

In [18]:
df_processed = df_combined.copy()

df_processed["ePixMean"] = df_processed["ePix"].apply(np.mean)
df_processed["levelMean"] = df_processed["level"].apply(np.mean)


In [19]:
skeleton_features = df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1)

fd_features = df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)


new_skeleton_features_df = pd.json_normalize(skeleton_features)
new_fd_features_df = pd.json_normalize(fd_features)


df_processed = pd.concat([
    df_processed.reset_index(drop=True),
    new_skeleton_features_df,
    new_fd_features_df,
], axis=1)


In [20]:
def plot_event_fd_full_from_row(row, n_coeffs=10, save_path=None):
    xPix = np.array(getattr(row, 'xPix'))
    yPix = np.array(getattr(row, 'yPix'))
    
    # --- Mask ---
    mask = create_mask_from_coords(xPix, yPix)
    
    # --- Skeleton & Features ---
    skeleton = skeletonize(mask)
    kernel = np.array([[1,1,1],[1,0,1],[1,1,1]])
    neighbors = convolve(skeleton.astype(np.uint8), kernel, mode='constant', cval=0)
    
    skeleton_pixels = skeleton.astype(bool)
    endpoints = (neighbors == 1) & skeleton_pixels
    branchpoints = (neighbors >= 3) & skeleton_pixels
    num_endpoints = endpoints.sum()
    num_branches = branchpoints.sum()
    
    # --- FD Magnitudes ---
    complex_contour = xPix + 1j*yPix
    coeffs = np.fft.fft(complex_contour)
    s1_mag = np.abs(coeffs[1]) if np.abs(coeffs[1])>0 else 1.0
    FDs = np.abs(coeffs[2:12]) / s1_mag

    if len(FDs) < 10:
        FDs = np.pad(FDs, (0, 10 - len(FDs)), 'constant')
    
    # --- FD Inverse FFT ---
    mask_contours = find_contours(mask, 0.5)
    if mask_contours:
        contour = max(mask_contours, key=len)
        y_c, x_c = contour[:,0], contour[:,1]
        complex_contour_mask = x_c + 1j*y_c
        coeffs_mask = np.fft.fft(complex_contour_mask)
        truncated = np.zeros_like(coeffs_mask)
        truncated[:n_coeffs+1] = coeffs_mask[:n_coeffs+1]
        truncated[-n_coeffs:] = coeffs_mask[-n_coeffs:]
        reconstructed = np.fft.ifft(truncated)
        recon_x, recon_y = reconstructed.real, reconstructed.imag
    else:
        recon_x, recon_y = np.array([]), np.array([])
    
    # --- Plot 1x4 inline ---
    fig, axes = plt.subplots(1, 4, figsize=(16,4), constrained_layout=False)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.92, bottom=0.15, wspace=0.3)
    
    # Panel (a) Mask
    axes[0].imshow(mask, cmap='gray')
    axes[0].set_title("Binary Mask", fontsize=16)
    axes[0].axis('off')
    
    # Panel (b) Skeleton
    axes[1].imshow(mask, cmap='gray', alpha=0.2)
    skel_y, skel_x = np.where(skeleton)
    end_y, end_x = np.where(endpoints)
    br_y, br_x = np.where(branchpoints)
    
    axes[1].scatter(skel_x, skel_y, s=20, color='red', label='Skeleton')
    axes[1].scatter(end_x, end_y, s=80, color='blue', marker='o', label=f'Endpoints ({num_endpoints})')
    axes[1].scatter(br_x, br_y, s=100, color='green', marker='s', label=f'Branchpoints ({num_branches})')
    axes[1].set_title("Skeleton with Features", fontsize=16)
    axes[1].axis('off')
    axes[1].legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),
        fontsize=12,
        ncol=3,
        handletextpad=0.3,   # reduce space between marker and text
        columnspacing=0.8    # reduce space between columns
    )
    axes[1].legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.01),  # closer to the panel
        fontsize=12,
        ncol=1,                        # vertical layout
        handletextpad=0.3,             # marker-text spacing
        labelspacing=0.3               # vertical spacing between items
    )
    # Panel (c) FD Magnitudes
    axes[2].bar(range(2,12), FDs, color='magenta')
    axes[2].set_xlabel("FD index", fontsize=14)
    axes[2].set_ylabel("Normalized magnitude", fontsize=14)
    axes[2].set_title("FD Magnitudes", fontsize=16)
    axes[2].tick_params(axis='both', labelsize=12)
    
    # Panel (d) FD Inverse FFT
    axes[3].imshow(mask, cmap='gray', alpha=0.2)
    if recon_x.size > 0:
        axes[3].plot(recon_x, recon_y, color='magenta', lw=2, label='FD Reconstruction')
        axes[3].set_title("FD-based Reconstruction", fontsize=16)
    axes[3].axis('off')
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

In [133]:
def plot_event_fd_full_from_row(row, n_coeffs=10, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.ndimage import convolve
    from skimage.morphology import skeletonize
    from skimage.measure import find_contours
    from scipy.spatial.distance import cdist

    # ---------------------------
    # FONT SCALE CONTROL (KEY FIX)
    # ---------------------------
    FONT = 1.35

    plt.rcParams.update({
        "font.size": int(16 * FONT),
        "axes.titlesize": int(20 * FONT),
        "axes.labelsize": int(16 * FONT),
        "xtick.labelsize": int(14 * FONT),
        "ytick.labelsize": int(14 * FONT),
        "legend.fontsize": int(13 * FONT)
    })

    # ---------------------------
    # DATA
    # ---------------------------
    xPix = np.array(getattr(row, 'xPix'))
    yPix = np.array(getattr(row, 'yPix'))

    mask = create_mask_from_coords(xPix, yPix)

    # ---------------------------
    # SKELETON
    # ---------------------------
    skeleton = skeletonize(mask)

    kernel = np.array([[1,1,1],
                       [1,0,1],
                       [1,1,1]])

    neighbors = convolve(skeleton.astype(np.uint8),
                         kernel,
                         mode='constant',
                         cval=0)

    endpoints = skeleton & (neighbors <= 1)
    branches  = skeleton & (neighbors >= 3)

    skel_y, skel_x = np.where(skeleton)
    end_y, end_x = np.where(endpoints)
    br_y, br_x = np.where(branches)

    num_endpoints = endpoints.sum()
    num_branches = branches.sum()

    # ---------------------------
    # FEATURES
    # ---------------------------
    skeleton_area_ratio = skeleton.sum() / (mask.sum() + 1e-8)

    coords = np.column_stack([skel_x, skel_y])

    if len(coords) >= 2:
        end_coords = np.column_stack([end_x, end_y])

        if len(end_coords) >= 2:
            dists = cdist(end_coords, end_coords)
            i, j = np.unravel_index(np.argmax(dists), dists.shape)
            p1, p2 = end_coords[i], end_coords[j]
        else:
            p1, p2 = coords[0], coords[-1]

        euclid = np.linalg.norm(p1 - p2)

        path_length = np.sum(np.linalg.norm(np.diff(coords, axis=0), axis=1))
        tortuosity = path_length / (euclid + 1e-8)

        line_vec = p2 - p1
        line_vec = line_vec / (np.linalg.norm(line_vec) + 1e-8)

        proj = coords - p1
        proj_len = proj @ line_vec
        proj_pts = np.outer(proj_len, line_vec) + p1
        linearity_dev = np.mean(np.linalg.norm(coords - proj_pts, axis=1))

        v1 = np.diff(coords[:-1], axis=0)
        v2 = np.diff(coords[1:], axis=0)

        def angle(a, b):
            na = np.linalg.norm(a, axis=1)
            nb = np.linalg.norm(b, axis=1)
            cos = np.sum(a*b, axis=1) / (na*nb + 1e-8)
            return np.arccos(np.clip(cos, -1, 1))

        curvature_sum = np.sum(np.abs(angle(v1, v2)))
    else:
        tortuosity = linearity_dev = curvature_sum = np.nan

    # ---------------------------
    # FOURIER DESCRIPTORS
    # ---------------------------
    c = xPix + 1j * yPix
    coeffs = np.fft.fft(c)

    s1 = np.abs(coeffs[1]) if np.abs(coeffs[1]) > 0 else 1.0
    FDs = np.abs(coeffs[2:12]) / s1

    if len(FDs) < 10:
        FDs = np.pad(FDs, (0, 10 - len(FDs)))

    # ---------------------------
    # RECONSTRUCTION
    # ---------------------------
    mask_contours = find_contours(mask, 0.5)

    rx, ry = np.array([]), np.array([])

    if mask_contours:
        contour = max(mask_contours, key=len)
        y_c, x_c = contour[:, 0], contour[:, 1]

        c2 = x_c + 1j * y_c
        cf = np.fft.fft(c2)

        tr = np.zeros_like(cf)
        tr[:n_coeffs + 1] = cf[:n_coeffs + 1]
        tr[-n_coeffs:] = cf[-n_coeffs:]

        r = np.fft.ifft(tr)
        rx, ry = r.real, r.imag

    # ---------------------------
    # FIGURE
    # ---------------------------
    fig = plt.figure(figsize=(20, 11))

    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1, 1],
        hspace=0.65,
        wspace=0.3
    )

    fig.subplots_adjust(top=0.93)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[0, 2])

    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1:3])

    # ---------------------------
    # MASK
    # ---------------------------
    ax0.imshow(mask, cmap='gray')
    ax0.set_title("Binary Mask")
    ax0.axis('off')

    # ---------------------------
    # SKELETON
    # ---------------------------
    ax1.imshow(mask, cmap='gray', alpha=0.2)

    ax1.scatter(skel_x, skel_y, s=10*FONT, color='red', alpha=0.6, label="Skeleton")
    ax1.scatter(end_x, end_y, s=70*FONT, color='blue',
                label=f"Endpoints ({num_endpoints})")
    ax1.scatter(br_x, br_y, s=80*FONT, color='green',
                label=f"Branchpoints ({num_branches})")

    ax1.set_title("Skeleton Structure")
    ax1.axis('off')

    ax1.legend(
        loc='lower center',
        bbox_to_anchor=(0.5, -0.40),
        ncol=2,
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor='black',
        fontsize=int(15 * FONT)
    )

    # ---------------------------
    # FEATURES
    # ---------------------------
    names = ["Branches", "Endpoints", "Area",
             "Tortuosity", "Linearity", "Curvature"]

    vals = np.array([
        branches.sum(),
        endpoints.sum(),
        skeleton_area_ratio,
        tortuosity,
        linearity_dev,
        curvature_sum
    ])

    bars = ax2.barh(names, vals, color="purple")
    ax2.set_xlim(0, vals.max() * 1.15)
    ax2.set_title("Skeleton Features")

    ax2.tick_params(labelsize=int(14 * FONT))

    for bar in bars:
        width = bar.get_width()
        ax2.text(width,
                 bar.get_y() + bar.get_height()/2,
                 f"{width:.2f}",
                 va='center',
                 ha='left',
                 fontsize=int(12 * FONT))

    # ---------------------------
    # FOURIER
    # ---------------------------
    ax3.bar(range(2, 12), FDs, color='magenta')
    ax3.set_title("Fourier Descriptor Magnitudes")
    ax3.set_xlabel("FD index")
    ax3.set_ylabel("Normalized magnitude")

    ax3.tick_params(labelsize=int(14 * FONT))

    # ---------------------------
    # RECONSTRUCTION
    # ---------------------------
    ax4.imshow(mask, cmap='gray', alpha=0.2)

    if rx.size > 0:
        ax4.plot(rx, ry, color='magenta', lw=2, label="FD reconstruction")

    ax4.set_title("FD-based Reconstruction")
    ax4.axis('off')

    # ---------------------------
    # OUTPUT
    # ---------------------------
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

In [134]:
# List of classes and number of samples per class
class_names_list = ['Blob', 'Diffusion Hit', 'Electron', 'Muon', 'Others']
n_samples = 4  # number of events per class to plot
# Loop over each class
for class_name in class_names_list:
    # Select the first n_samples from each class
    class_samples = df_combined[df_combined['label'] == class_name].head(n_samples)
    features_img_savepath = os.path.join(OUTPUT_FOLDER, "features_example", class_name)
    os.makedirs(features_img_savepath, exist_ok=True)
    for i, sample in enumerate(class_samples.itertuples(index=False), start=1):
        # Construct a filename
        save_path = os.path.join(features_img_savepath,  f"{class_name.lower().replace(' ', '_')}_event_{i}.png")
        # Plot and save the 4-panel figure
        plot_event_fd_full_from_row(sample, n_coeffs=10, save_path=save_path)


In [ ]:
df_processed = df_processed.drop(columns=["label", "xPix", "yPix", "level", "ePix", "flag"])

# Drop columns with no variance
df_processed = df_processed.loc[:, df_processed.nunique() > 1]

In [ ]:
print(df_processed.columns)

In [ ]:
print(df_processed)

Calculating the correlation between features

In [ ]:
corr_df_combined = df_processed.corr()
corr_pairs = corr_df_combined.unstack()
# Filter out self-correlations
filtered = corr_pairs[corr_pairs != 1.0]
# Remove duplicate mirror entries
filtered = filtered.drop_duplicates()
# Find correlations above 0.9
high_corr = filtered[filtered.abs() > 0.9]
print(high_corr.sort_values(ascending=False))

In [ ]:
print(df_processed.columns)

Removing features from the dataframe

In [ ]:
drop_cols = [
    # ===== DETECTOR METADATA (Not event-specific) =====
    "ohdu",        # Duplicate of skpID (correlation = 1.0 with skpID)
    "chid",        # Duplicate of skpID (correlation = -1.0)
    "skpID",       # Sensor ID - same for all events from same sensor
    "runID",       # Run ID - same for all events in same run
    "imgID",       # Image ID - same for all events in same image
    "Gain",        # Global gain - same for all events in same run
    
    # ===== IMAGE-LEVEL METADATA (Same for all events in image) =====
    "expoStart",   # Exposure start timestamp
    "DeltaT",      # Readout time
    "NpixAC",      # Number of unmasked active pixels
    
    # ===== EXACT DUPLICATES - Keep Level 0, Drop Level 1 =====
    "E1",          # correlation = 1.000 with E0
    "n1",          # correlation = 0.998 with n0
    "xBary1",      # correlation = 1.000 with xBary0
    "yBary1",      # correlation = 1.000 with yBary0
    "xVar1",       # correlation = 1.000 with xVar0
    "yVar1",       # correlation = 1.000 with yVar0
    
    # ===== REDUNDANT SIZE FEATURES (All ≈ n0, correlation ~0.998) =====
    "nSavedPix",   # Total saved pixels
    "nxPix",       # Length of xPix array
    "nyPix",       # Length of yPix array
    "nlevel",      # Number of levels
    "nePix",       # Number of energy pixels
    
    # ===== HIGH CORRELATION WITH BARYCENTER (>0.995) =====
    "xMin",        # correlation = 0.996 with xBary0
    "xMax",        # correlation = 0.996 with xBary0
    "yMin",        # correlation = 0.999 with yBary0
    "yMax",        # correlation = 0.999 with yBary0
    
    # ===== HIGH CORRELATION WITH OTHER FEATURES (>0.96) =====
    "skeleton_length",      # correlation = 0.963 with n0
    "branch_to_end_ratio",  # correlation = 0.966 with num_branches
]


df_processed_final = df_processed.drop(columns=drop_cols)


In [ ]:
print(df_processed_final.columns)

* Set all classes other than the current class to label 0
* Split the data into training and test sets
* Use k-fold cross-validation for training and validation

## Hyperparameters Tuning

In [ ]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
# mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

x_train = df_processed_final.copy()
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_combined["label"])

for i, class_name in enumerate(label_encoder.classes_):
    print(f"Class ID {i}: {class_name}")

k_folds = 5
kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed.get_seed())

In [ ]:
metrics_dir_xgboost = os.path.join(RESULTS_FOLDER, "metrics_xgboostcs_img_descriptor_new_features")
os.makedirs(metrics_dir_xgboost, exist_ok=True)

metrics_dir_rf = os.path.join(RESULTS_FOLDER, "metrics_rf_multiclass_img_descriptor_new_features")
os.makedirs(metrics_dir_rf, exist_ok=True)

In [ ]:
def cross_val_score_with_logging(model, x_train_cv, y_train_cv,
                                 trial_number, is_xgboost=False):

    val_scores = []
    train_scores = []

    val_f1_scores, val_precision_scores, val_recall_scores = [], [], []
    train_f1_scores, train_precision_scores, train_recall_scores = [], [], []

    all_val_preds = []
    all_val_true = []
    all_train_preds = []
    all_train_true = []

    if is_xgboost:
        metrics_dir = metrics_dir_xgboost
    else:
        metrics_dir = metrics_dir_rf

    for train_idx, val_idx in kf.split(x_train_cv, y_train_cv):
        fold_model = clone(model)
        x_tr, x_val = x_train_cv.iloc[train_idx], x_train_cv.iloc[val_idx]
        y_tr, y_val = y_train_cv[train_idx], y_train_cv[val_idx]

        scaler = StandardScaler()
        x_tr_scaled = scaler.fit_transform(x_tr)
        x_val_scaled = scaler.transform(x_val)

        x_tr_scaled = pd.DataFrame(x_tr_scaled, columns=x_tr.columns, index=x_tr.index)
        x_val_scaled = pd.DataFrame(x_val_scaled, columns=x_val.columns, index=x_val.index)

        if is_xgboost:
            classes = np.unique(y_tr)
            class_weights = compute_class_weight('balanced', classes=classes, y=y_tr)
            weight_dict = dict(zip(classes, class_weights))
            sample_weights = np.array([weight_dict[y] for y in y_tr])
            fold_model.fit(x_tr_scaled, y_tr, sample_weight=sample_weights)
        else:
            fold_model.fit(x_tr_scaled, y_tr)

        y_tr_pred = fold_model.predict(x_tr_scaled)
        y_val_pred = fold_model.predict(x_val_scaled)

        train_acc = accuracy_score(y_tr, y_tr_pred)
        val_acc = accuracy_score(y_val, y_val_pred)

        train_scores.append(train_acc)
        val_scores.append(val_acc)

        train_f1_scores.append(f1_score(y_tr, y_tr_pred, average="macro", zero_division=0))
        train_precision_scores.append(precision_score(y_tr, y_tr_pred, average="macro", zero_division=0))
        train_recall_scores.append(recall_score(y_tr, y_tr_pred, average="macro", zero_division=0))

        val_f1_scores.append(f1_score(y_val, y_val_pred, average="macro", zero_division=0))
        val_precision_scores.append(precision_score(y_val, y_val_pred, average="macro", zero_division=0))
        val_recall_scores.append(recall_score(y_val, y_val_pred, average="macro", zero_division=0))

        all_val_preds.extend(y_val_pred)
        all_val_true.extend(y_val)
        all_train_preds.extend(y_tr_pred)
        all_train_true.extend(y_tr)

    val_mean_acc = np.mean(val_scores)
    val_std_acc = np.std(val_scores)
    train_mean_acc = np.mean(train_scores)

    val_mean_f1 = np.mean(val_f1_scores)
    val_std_f1 = np.std(val_f1_scores)

    val_mean_precision = np.mean(val_precision_scores)
    val_std_precision = np.std(val_precision_scores)

    val_mean_recall = np.mean(val_recall_scores)
    val_std_recall = np.std(val_recall_scores)

    # === Reports (unchanged) ===
    val_report = classification_report(
        all_val_true, all_val_preds,
        target_names=label_encoder.classes_,
        output_dict=True, zero_division=0
    )
    val_report_df = pd.DataFrame(val_report).transpose()

    report_path = os.path.join(metrics_dir,
                              f"trial_{trial_number}_classification_report.csv")
    val_report_df.to_csv(report_path)
    mlflow.log_artifact(report_path)

    # Confusion matrix (unchanged)
    cm = confusion_matrix(
        all_val_true, all_val_preds,
        labels=np.arange(len(label_encoder.classes_))
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_, ax=ax)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()

    cm_path = os.path.join(metrics_dir,
                           f"trial_{trial_number}_confusion_matrix.png")
    fig.savefig(cm_path, dpi=150)
    mlflow.log_artifact(cm_path)
    plt.close(fig)

    return (
        (train_mean_acc, np.mean(train_f1_scores),
         np.mean(train_precision_scores), np.mean(train_recall_scores)),
        (val_mean_acc, val_mean_f1,
         val_mean_precision, val_mean_recall),
        (val_std_acc, val_std_f1, val_std_precision, val_std_recall)
    )

In [ ]:
def objective_random_forest(trial, x_train_cv, y_train_cv):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 3, 10),
        "random_state": seed.get_seed(),
        "class_weight": "balanced"
    }

    model = RandomForestClassifier(**params)

    with mlflow.start_run(nested=True, run_name=f"RF_trial_{trial.number}"):

        (train_means, val_means, val_stds) = cross_val_score_with_logging(
            model, x_train_cv, y_train_cv, trial.number)

        (train_acc, train_f1_macro,
         train_precision, train_recall) = train_means

        (val_acc, val_f1_macro,
         val_precision, val_recall) = val_means

        (val_std_acc, val_std_f1,
         val_std_precision, val_std_recall) = val_stds

        mlflow.set_tag("model_type", "RandomForest")
        mlflow.set_tag("kfold_splits", 5)
        mlflow.log_params(params)

        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "train_precision": train_precision,
            "train_recall": train_recall,

            "val_accuracy": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "val_std_accuracy": val_std_acc,
            "val_std_f1": val_std_f1,
            "val_std_precision": val_std_precision,
            "val_std_recall": val_std_recall,
        })


        full_metrics = {
            "train": {
                "accuracy": train_acc,
                "f1_macro": train_f1_macro,
                "precision": train_precision,
                "recall": train_recall,
            },
            "validation": {
                "accuracy_mean": val_acc,
                "accuracy_std": val_std_acc,
                "f1_mean": val_f1_macro,
                "f1_std": val_std_f1,
                "precision_mean": val_precision,
                "precision_std": val_std_precision,
                "recall_mean": val_recall,
                "recall_std": val_std_recall,
            },
            "params": params,
            "trial_number": trial.number
        }

        mlflow.log_dict(full_metrics, "full_metrics.json")

    return val_f1_macro

In [ ]:
def objective_xgboost(trial, x_train_cv, y_train_cv):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "use_label_encoder": False,
        "eval_metric": "mlogloss",
        "random_state": seed.get_seed(),
        "objective": "multi:softprob",
        "num_class": len(np.unique(y_train_cv)),
    }

    model = XGBClassifier(**params)

    with mlflow.start_run(nested=True, run_name=f"XGB_trial_{trial.number}"):

        (train_means, val_means, val_stds) = cross_val_score_with_logging(
            model, x_train_cv, y_train_cv, trial.number, is_xgboost=True)

        (train_acc, train_f1_macro,
         train_precision, train_recall) = train_means

        (val_acc, val_f1_macro,
         val_precision, val_recall) = val_means

        (val_std_acc, val_std_f1,
         val_std_precision, val_std_recall) = val_stds

        mlflow.set_tag("model_type", "XGBoost")
        mlflow.set_tag("kfold_splits", 5)
        mlflow.log_params(params)

        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "val_accuracy": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_std_accuracy": val_std_acc,
            "val_std_f1": val_std_f1,
            "val_std_precision": val_std_precision,
            "val_std_recall": val_std_recall,
        })

        full_metrics = {
            "train": {
                "accuracy": train_acc,
                "f1_macro": train_f1_macro,
                "precision": train_precision,
                "recall": train_recall,
            },
            "validation": {
                "accuracy_mean": val_acc,
                "accuracy_std": val_std_acc,
                "f1_mean": val_f1_macro,
                "f1_std": val_std_f1,
                "precision_mean": val_precision,
                "precision_std": val_std_precision,
                "recall_mean": val_recall,
                "recall_std": val_std_recall,
            },
            "params": params,
            "trial_number": trial.number
        }

        mlflow.log_dict(full_metrics, "full_metrics.json")

    return val_f1_macro

In [ ]:
def objective_xgb_wrapped(trial):
    return objective_xgboost(trial, x_train_cv=x_train, y_train_cv=y_train)

def objective_rf_wrapped(trial):
    return objective_random_forest(trial, x_train_cv=x_train, y_train_cv=y_train)


In [ ]:
mlflow.set_experiment(f"Tuning_XGB_multiclass_img_descriptor_new_features_fixed_2")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb_wrapped, n_trials=100)

mlflow.set_experiment(f"Tuning_RF_multiclass_img_descriptor_new_features_fixed_2")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf_wrapped, n_trials=100)


In [ ]:
best_params_rf = study_rf.best_trial.params

In [ ]:
best_params_xgb = study_xgb.best_trial.params